# 05 — Comparacao de Modelos de Embedding

Qual modelo escolher para o seu projeto RAG?

| Modelo | Dims | Parametros | Qualidade MTEB | Velocidade |
|--------|------|-----------|----------------|------------|
| all-MiniLM-L6-v2 | 384 | 22M | 56.3 | ⚡⚡⚡ |
| all-mpnet-base-v2 | 768 | 110M | 57.8 | ⚡⚡ |
| all-roberta-large-v1 | 1024 | 355M | 60.0 | ⚡ |
| nomic-embed-text (via Ollama) | 768 | 137M | 62.0 | ⚡⚡ |

**O que vamos ver:**
1. Benchmark de velocidade
2. Qualidade semantica comparada
3. Como usar via Ollama (modelo local)
4. Guia de decisao

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sentence_transformers import SentenceTransformer

MODELOS = [
    {'name': 'all-MiniLM-L6-v2',   'dims': 384,  'params_m': 22,  'cor': '#3498db'},
    {'name': 'all-mpnet-base-v2',  'dims': 768,  'params_m': 110, 'cor': '#2ecc71'},
    # Descomente para incluir o modelo maior (~1.3GB):
    # {'name': 'all-roberta-large-v1', 'dims': 1024, 'params_m': 355, 'cor': '#e74c3c'},
]

# Dataset de avaliacao
queries = [
    'Como funciona o algoritmo de backpropagation?',
    'Quais sao as melhores praticas de seguranca web?',
    'Como preparar uma boa massa de pizza caseira?',
    'Qual e a diferenca entre supervised e unsupervised learning?',
    'Onde fica o Cristo Redentor no Rio de Janeiro?',
]

docs = [
    'Backpropagation calcula gradientes pela regra da cadeia nas redes neurais.',  # Q0
    'Gradient descent otimiza parametros iterativamente para minimizar o loss.',  # Q0, Q3
    'HTTPS usa SSL/TLS para criptografar dados em transito.',                    # Q1
    'SQL injection e um ataque que manipula queries de banco de dados.',          # Q1
    'A massa de pizza deve descansar 1 hora para o gluten se desenvolver.',       # Q2
    'Farinha 00 italiana e ideal para pizza napolitana autentica.',               # Q2
    'Supervised learning usa dados rotulados; unsupervised descobre padroes.',    # Q3
    'K-means e um algoritmo de clustering nao supervisionado popular.',           # Q3
    'O Cristo Redentor fica no Corcovado, Rio de Janeiro, com 38 metros.',        # Q4
    'A Amazonia e o maior bioma do Brasil em extensao territorial.',              # irrelevante
]

# Ground truth: qual doc e mais relevante para cada query (idx)
ground_truth = {0: 0, 1: 2, 2: 4, 3: 6, 4: 8}

print(f'{len(queries)} queries, {len(docs)} documentos')

## 5.1 Benchmark de Velocidade

In [ ]:
# Textos para benchmark de throughput
import random
random.seed(42)
bench_texts = [f'Este e o documento numero {i} sobre {random.choice(["IA", "dados", "Python", "cloud"])}.' 
               for i in range(1000)]

bench_results = {}

for info in MODELOS:
    print(f'\nBenchmark: {info["name"]}...')
    m = SentenceTransformer(info['name'])
    
    # Warm-up
    _ = m.encode(['warm up'], show_progress_bar=False)
    
    start = time.perf_counter()
    embs = m.encode(bench_texts, batch_size=64, show_progress_bar=False)
    elapsed = time.perf_counter() - start
    
    texts_per_sec = len(bench_texts) / elapsed
    
    bench_results[info['name']] = {
        'dims': info['dims'],
        'params_m': info['params_m'],
        'texts_per_sec': texts_per_sec,
        'time_s': elapsed,
        'cor': info['cor'],
    }
    print(f'  {texts_per_sec:.0f} textos/sec ({elapsed:.2f}s para 1000 textos)')
    del m

## 5.2 Avaliacao de Qualidade Semantica

In [ ]:
quality_results = {}

for info in MODELOS:
    m = SentenceTransformer(info['name'])
    
    q_embs = m.encode(queries, normalize_embeddings=True)
    d_embs = m.encode(docs, normalize_embeddings=True)
    
    sim_matrix = q_embs @ d_embs.T  # (n_queries, n_docs)
    
    # Calcular metricas
    hits_at_1 = 0
    mrr = 0.0
    
    print(f'\n{"="*50}')
    print(f'Modelo: {info["name"]} ({info["dims"]}d)')
    print(f'{"="*50}')
    
    for q_idx, query in enumerate(queries):
        scores = sim_matrix[q_idx]
        ranked_docs = np.argsort(scores)[::-1]
        expected_doc = ground_truth[q_idx]
        rank = np.where(ranked_docs == expected_doc)[0][0] + 1
        
        if rank == 1:
            hits_at_1 += 1
        mrr += 1.0 / rank
        
        print(f'Q{q_idx}: "{query[:40]}..."')
        print(f'  Top-1: doc[{ranked_docs[0]}] = "{docs[ranked_docs[0]][:60]}"')
        print(f'  Rank do doc correto: {rank} | Score correto: {scores[expected_doc]:.3f}')
    
    mrr /= len(queries)
    hit_rate = hits_at_1 / len(queries)
    
    quality_results[info['name']] = {
        'hit_at_1': hit_rate,
        'mrr': mrr,
        'dims': info['dims'],
    }
    print(f'\nHit@1: {hit_rate:.2%} | MRR: {mrr:.3f}')
    del m

In [ ]:
# Tabela resumo comparativa
rows = []
for nome, br in bench_results.items():
    qr = quality_results.get(nome, {})
    rows.append({
        'Modelo': nome,
        'Dims': br['dims'],
        'Params (M)': br['params_m'],
        'Throughput (textos/s)': f"{br['texts_per_sec']:.0f}",
        'Hit@1': f"{qr.get('hit_at_1', 0):.0%}",
        'MRR': f"{qr.get('mrr', 0):.3f}",
    })

df = pd.DataFrame(rows)
print('Resumo Comparativo:\n')
print(df.to_string(index=False))

In [ ]:
# Visualizacao
nomes = list(bench_results.keys())
cores = [bench_results[n]['cor'] for n in nomes]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Throughput
speeds = [bench_results[n]['texts_per_sec'] for n in nomes]
bars = axes[0].bar(range(len(nomes)), speeds, color=cores)
axes[0].set_xticks(range(len(nomes)))
axes[0].set_xticklabels([n.split('/')[-1] for n in nomes], rotation=30, ha='right', fontsize=9)
axes[0].set_title('Throughput (textos/segundo)', fontweight='bold')
axes[0].set_ylabel('Textos/segundo')
for bar, speed in zip(bars, speeds):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{speed:.0f}', ha='center', va='bottom', fontweight='bold')

# Scatter: Speed vs Quality
for i, nome in enumerate(nomes):
    axes[1].scatter(
        bench_results[nome]['texts_per_sec'],
        quality_results.get(nome, {}).get('mrr', 0),
        s=200, color=cores[i], label=nome.split('/')[-1], zorder=3
    )
    axes[1].annotate(nome.split('/')[-1],
                    (bench_results[nome]['texts_per_sec'],
                     quality_results.get(nome, {}).get('mrr', 0)),
                    xytext=(5, 5), textcoords='offset points', fontsize=8)

axes[1].set_title('Speed vs Quality Trade-off\n(ideal: alto e a direita)', fontweight='bold')
axes[1].set_xlabel('Throughput (textos/s)')
axes[1].set_ylabel('MRR (Mean Reciprocal Rank)')
axes[1].grid(True, alpha=0.3)

# Comparacao de dimensoes
dims = [bench_results[n]['dims'] for n in nomes]
axes[2].barh(range(len(nomes)), dims, color=cores)
axes[2].set_yticks(range(len(nomes)))
axes[2].set_yticklabels([n.split('/')[-1] for n in nomes], fontsize=9)
axes[2].set_title('Dimensoes dos Vetores', fontweight='bold')
axes[2].set_xlabel('Numero de dimensoes')
for i, d in enumerate(dims):
    axes[2].text(d + 5, i, str(d), va='center', fontweight='bold')

plt.suptitle('Comparacao de Modelos de Embedding', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5.3 Usando Embeddings via Ollama

O Ollama pode servir modelos de embedding localmente, sem Python weights.
Util quando voce ja tem Ollama rodando para o LLM.

In [ ]:
import ollama
import httpx

# Verificar se nomic-embed-text esta disponivel
try:
    r = httpx.get('http://localhost:11434/api/tags')
    models_available = [m['name'] for m in r.json().get('models', [])]
    print(f'Modelos Ollama disponiveis: {models_available}')
    
    if 'nomic-embed-text' not in str(models_available):
        print('\nnomic-embed-text nao encontrado.')
        print('Execute: docker exec ollama ollama pull nomic-embed-text')
    else:
        # Testar embedding via Ollama
        response = ollama.embeddings(
            model='nomic-embed-text',
            prompt='Como funciona machine learning?'
        )
        vec = response['embedding']
        print(f'\nnomic-embed-text embedding: {len(vec)} dimensoes')
        print(f'Primeiros 5 valores: {vec[:5]}')
        
except Exception as e:
    print(f'Ollama nao disponivel: {e}')
    print('Execute: docker compose up -d ollama')

## Guia de Decisao: Qual modelo escolher?

```
Seu caso de uso?
│
├── Prototipagem / aprendizagem
│   └── all-MiniLM-L6-v2 (384d)
│       Mais rapido, menor, facil de usar
│
├── Producao generalista
│   └── all-mpnet-base-v2 (768d)
│       Melhor qualidade/velocidade, padrao SBERT
│
├── Qualidade maxima local
│   └── all-roberta-large-v1 (1024d)
│       Mais lento, mais preciso, para dados exigentes
│
├── Ja usando Ollama
│   └── nomic-embed-text (768d) ou mxbai-embed-large (1024d)
│       Sem dependencia adicional no Python
│
└── Multilingual
    └── paraphrase-multilingual-mpnet-base-v2 (768d)
        Suporta 50+ linguas
```

## Proximos passos
- [Modulo 02 — Qdrant](../02_vector_databases/README.md): Agora que entendemos embeddings, vamos indexar!
- [docs/embeddings/models_overview.md](../docs/embeddings/models_overview.md): Tabela completa de modelos